# YOLO V5 — seven-class training, with Worker A/B/C reviewed annotation patches

**Project:** cafeteria waste object detection · `yolo11s.pt` · seven classes.

This notebook starts from the **latest Batch 4 dataset** (`YOLO_V5_partial_annotation_audit320.zip`) and applies completed annotation patches from Workers A, B, and C **without duplicating images**. All **1,252 assigned images already exist** in the 2,093-image V5 training set; their **labels** are what must be replaced after human/AI review.

| Dataset | Train | Validation | Test |
|---|---:|---:|---:|
| V5 baseline | **2,093** | **97** | **71** |

**Before training:** 320 original images have provisional visual audits. **1,252** assigned originals still await completed worker reviews. Worker input ZIPs are **NOT corrected patches**. By default, this notebook **refuses training** until all 1,252 images have signed-off review logs. You can explicitly allow exploratory training with incomplete labels, but the results must not be described as seven-class validated.

Class order: **0** food · **1** plate · **2** utensil · **3** hinged_container · **4** drink_carton · **5** wrapper · **6** bottle.

⚠️ **Evaluation limitation:** The existing 97-image validation split has **zero labeled hinged containers, cartons or bottles** and only **13 wrappers**. It cannot support valid seven-class mAP claims. V4 images also have unresolved cross-class annotation omissions. This notebook does **not** touch the reserved 71-image test split.

## 1. Prepare files in Google Drive

Create the Drive folder **`MyDrive/YOLO_V5`**. Put these files there:

- **`YOLO_V5_partial_annotation_audit320.zip`** — use the **Batch 4** complete V5 archive, not the original Day1 archive.
- **`MASTER_ASSIGNMENT_INDEX.csv`** — from the Command Center control ZIP.
- **`reviewed_patches/`** — put worker-produced reviewed **batch ZIPs** here *as they arrive*. Do **not** place the original `V5_Worker_A_402_Pending_Images.zip` / B / C input ZIPs here.

### Required format for each reviewed worker batch ZIP

- Corrected YOLO text labels, one for **every** reviewed image, including unchanged labels. Recommended structure: `labels/train/<original_filename>.txt`. Also accepted: `<anything>/after_labels/<original_filename>.txt` or `<anything>/corrected_labels/<original_filename>.txt`.
- **`review_manifest.csv`** with these columns:
  `worker,label_path,status,original_label_sha256,qc_status`
- Each row must use the canonical `labels/train/...txt` label path from `MASTER_ASSIGNMENT_INDEX.csv`, its expected **original** label SHA256, a worker ID (`A`, `B`, `C`), `status=REVIEWED` (or `REVIEWED_UNCHANGED`), and `qc_status=PASS`.
- Uncertain/flagged images must be resolved before `PASS`. Worker ZIPs with only provisional source labels or unverified status do not qualify.

The attached *Worker Handoff Templates* ZIP contains the three per-worker `review_manifest.csv` templates. Each worker can fill in the status columns after QC. Multiple 100-image patch ZIPs per worker are supported.

Colab: **Runtime → Change runtime type → T4 GPU** (or another available NVIDIA GPU). This notebook trains in Colab, not on your Mac.

In [ ]:
%pip -q install "ultralytics>=8.3,<9" pyyaml pillow

In [ ]:
from google.colab import drive
from pathlib import Path
import csv, hashlib, io, json, math, os, shutil, zipfile
from collections import Counter
from datetime import datetime, timezone
import yaml

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/YOLO_V5')
BASE_ZIP = ROOT / 'YOLO_V5_partial_annotation_audit320.zip'
MASTER_CSV = ROOT / 'MASTER_ASSIGNMENT_INDEX.csv'
PATCH_DIR = ROOT / 'reviewed_patches'
WORK_DIR = Path('/content')
DATASET_DIR = WORK_DIR / 'v5_original_plus_v4'

# Strong default: final training only after every assigned image is visually reviewed and QC-passed.
REQUIRE_ALL_1252_REVIEWED = True
ALLOW_PROVISIONAL_TRAINING = False  # Set True ONLY for an explicitly exploratory run.

# Training choices. All output files are saved to Google Drive.
MODEL_WEIGHTS = 'yolo11s.pt'
EPOCHS = 60
IMAGE_SIZE = 800
BATCH_SIZE = 4             # Fits typical Colab T4 GPUs more reliably at 800 px.
RUN_NAME = 'V5_7class_merged_worker_reviews'

print('Base dataset: ', BASE_ZIP)
print('Assignment CSV:', MASTER_CSV)
print('Worker patches:', PATCH_DIR)


## 2. Validate baseline, assignment ownership, and extract once

This cell checks the exact reviewed Batch 4 archive checksum, the 402/425/425 worker partition, all source image hashes and baseline label hashes, and the unchanged validation/test split. It **fails closed** on missing or incorrectly named data.

In [ ]:
BASE_SHA256 = '4410e575f3797acca490c240fd58b85113aefb1261157c24c257b5dbeb2ecd9b'
EXPECTED_WORKERS = {'A': 402, 'B': 425, 'C': 425}
CLASSES = ['food', 'plate', 'utensil', 'hinged_container', 'drink_carton', 'wrapper', 'bottle']


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(p):
    digest = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def read_csv_file(path):
    with open(path, newline='', encoding='utf-8-sig') as f:
        return list(csv.DictReader(f))


assert BASE_ZIP.is_file(), f'Missing {BASE_ZIP}. Copy the Batch 4 full ZIP to Drive.'
assert MASTER_CSV.is_file(), f'Missing {MASTER_CSV}. Extract it from Command Center control files.'
assert sha256_file(BASE_ZIP) == BASE_SHA256, 'Wrong V5 dataset version or corrupted ZIP; use the Batch 4 audit320 archive.'

assignments = read_csv_file(MASTER_CSV)
assert len(assignments) == 1252
assert Counter(r['worker'] for r in assignments) == EXPECTED_WORKERS
allowed = {r['label_path']: r for r in assignments}
assert len(allowed) == 1252, 'Duplicate label paths across worker assignments.'
assert all(r['status'] == 'PENDING' for r in assignments)
assert all(r['label_path'].startswith('labels/train/') for r in assignments)

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
with zipfile.ZipFile(BASE_ZIP) as z:
    assert z.testzip() is None, 'Source ZIP CRC failure.'
    for item in z.infolist():
        # Avoid unsafe archive paths. Dataset root is fixed.
        name = item.filename.replace('\\', '/')
        parts = Path(name).parts
        assert parts and parts[0] == 'v5_original_plus_v4', f'Unexpected ZIP member: {name}'
        assert '..' not in parts and not Path(name).is_absolute(), f'Unsafe ZIP entry: {name}'
    z.extractall(WORK_DIR)

with open(DATASET_DIR / 'manifest.json', encoding='utf-8') as f:
    full_manifest = json.load(f)
assert len(full_manifest) == 2261
assert Counter(r['split'] for r in full_manifest) == {'train': 2093, 'val': 97, 'test': 71}
manifest_by_label = {r['label_path']: r for r in full_manifest}
assert len(manifest_by_label) == len(full_manifest)

for entry in full_manifest:
    for kind in ('image', 'label'):
        p = DATASET_DIR / entry[kind + '_path']
        assert p.is_file(), f'Missing {kind}: {p}'
        assert sha256_file(p) == entry[kind + '_sha256'], f'Baseline {kind} checksum mismatch: {p}'

for path, assignment in allowed.items():
    row = manifest_by_label[path]
    assert row['split'] == 'train'
    assert row['label_sha256'] == assignment['original_label_sha256']
    assert row['image_sha256'] == assignment['original_image_sha256']

reserved_hashes = {
    r['label_path']: (r['image_sha256'], r['label_sha256'])
    for r in full_manifest if r['split'] in ('val', 'test')
}
print('✅ Verified baseline: 2,093 train / 97 val / 71 test')
print('✅ Worker assignments:', EXPECTED_WORKERS, '— 1,252 unique pending originals')
print('✅ Image and label hashes verified for all 2,261 images')


## 3. Apply **reviewed** worker label patches (never upload the original worker-assignment ZIPs here)

Each worker patch must contain `review_manifest.csv` **with explicit QC approval**, and one reviewed label file per approved manifest row. The importer checks worker ownership, original label hash, class IDs, box geometry, duplicates, and repeated assignments. Unknown files and validation/test changes are rejected. You can mix 100-image patches from multiple workers.

In [ ]:
APPROVED_STATUSES = {'REVIEWED', 'REVIEWED_UNCHANGED', 'VISUAL_REVIEW_REVISED', 'VISUAL_REVIEW_NO_CHANGE'}


def validate_yolo(raw, label_path):
    text = raw.decode('utf-8-sig')
    seen = set()
    counts = Counter()
    for lineno, line in enumerate(text.splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        values = line.split()
        assert len(values) == 5, f'{label_path}:{lineno}: expected 5 tokens'
        assert values[0] in {str(i) for i in range(7)}, f'{label_path}:{lineno}: unknown class'
        nums = tuple(float(x) for x in values[1:])
        assert all(math.isfinite(x) for x in nums), f'{label_path}:{lineno}: NaN/Inf'
        cx, cy, w, h = nums
        assert 0 < w <= 1 and 0 < h <= 1
        assert -0.00001 <= cx - w / 2 <= cx + w / 2 <= 1.00001
        assert -0.00001 <= cy - h / 2 <= cy + h / 2 <= 1.00001
        rounded = (values[0],) + tuple(round(v, 6) for v in nums)
        assert rounded not in seen, f'{label_path}:{lineno}: duplicate bounding box'
        seen.add(rounded)
        counts[int(values[0])] += 1
    return counts


def get_zip_member_for_label(z, canonical):
    name = Path(canonical).name
    candidates = []
    for member in z.namelist():
        p = Path(member)
        if p.name != name or not member.lower().endswith('.txt'):
            continue
        parents = [part.lower() for part in p.parts[:-1]]
        # Explicitly reject before/old labels when the archive contains both.
        if any(('before' in part or 'old_label' in part) for part in parents):
            continue
        if any('label' in part for part in parents):
            candidates.append(member)
    assert len(candidates) == 1, (f'{canonical}: expected ONE final label inside patch ZIP, '
                                  f'found {len(candidates)}: {candidates}')
    return candidates[0]


PATCH_DIR.mkdir(parents=True, exist_ok=True)
patches = sorted(PATCH_DIR.glob('*.zip'))
changes = {}                 # canonical label path -> replacement bytes
review_info = {}             # canonical label path -> audit record
patch_log = []

for patch_path in patches:
    with zipfile.ZipFile(patch_path) as z:
        assert z.testzip() is None, f'Corrupted patch ZIP: {patch_path.name}'
        manifests = [n for n in z.namelist() if Path(n).name == 'review_manifest.csv']
        assert len(manifests) == 1, (f'{patch_path.name} requires exactly one review_manifest.csv. '
                                      'Raw assigned-image ZIPs are NOT reviewed patches.')
        rows = list(csv.DictReader(io.StringIO(z.read(manifests[0]).decode('utf-8-sig'))))
        assert rows, f'Empty review manifest: {patch_path.name}'
        required = {'worker', 'label_path', 'status', 'original_label_sha256', 'qc_status'}
        assert required.issubset(rows[0].keys()), f'{patch_path.name} missing {required - set(rows[0])}'
        for row in rows:
            status = row['status'].strip().upper()
            if status not in APPROVED_STATUSES:
                # PENDING or FLAGGED rows in a complete-template file do not count.
                continue
            label = row['label_path'].strip()
            assert label in allowed, f'Unknown/non-training label in patch: {label}'
            owner = allowed[label]['worker']
            assert row['worker'].strip().upper() == owner, f'Wrong worker for {label}'
            assert row['qc_status'].strip().upper() == 'PASS', f'QC not passed for {label}'
            assert row['original_label_sha256'].strip().lower() == allowed[label]['original_label_sha256'], (
                f'Old-label hash mismatch in {patch_path.name}: {label}')
            assert label not in changes, f'{label} appears in multiple reviewed patches'
            data = z.read(get_zip_member_for_label(z, label))
            counts = validate_yolo(data, label)
            changes[label] = data
            review_info[label] = {
                'worker': owner, 'status': status, 'qc_status': 'PASS',
                'patch_zip': patch_path.name,
                'base_label_sha256': allowed[label]['original_label_sha256'],
                'final_label_sha256': sha256_bytes(data),
                'box_count': sum(counts.values()),
            }
    patch_log.append({'zip': patch_path.name, 'approved_total_so_far': len(changes)})

# Verify all old hashes BEFORE applying any patch, then apply atomically to working copy.
for label, raw in changes.items():
    file_path = DATASET_DIR / label
    assert sha256_file(file_path) == allowed[label]['original_label_sha256'], f'Stale base label: {label}'

for label, raw in changes.items():
    (DATASET_DIR / label).write_bytes(raw)
    m = manifest_by_label[label]
    m['label_sha256'] = sha256_bytes(raw)
    m['instances'] = sum(validate_yolo(raw, label).values())

counts_by_worker = Counter(review_info[x]['worker'] for x in review_info)
missing_by_worker = {w: EXPECTED_WORKERS[w] - counts_by_worker[w] for w in EXPECTED_WORKERS}
print('Reviewed patch ZIPs:', len(patches))
print('Approved reviewed labels applied:', len(changes), '/ 1252')
print('Progress per worker:', dict(counts_by_worker))
print('Still pending per worker:', missing_by_worker)
if not changes:
    print('⚠️ No completed reviewed worker patches supplied. Baseline remains provisional.')


## 4. Final quality gate, rewrite dataset bookkeeping, and show label distribution

This does not assert the annotations are visually perfect: it verifies file integrity, coordinates, known worker ownership, and documented signoff. A second independent visual adjudication is still advisable.

In [ ]:
status_path = DATASET_DIR / 'original_review_status.csv'
status_rows = read_csv_file(status_path)
assert len(status_rows) == 1572
status_by_label = {r['label_path']: r for r in status_rows}

for label, info in review_info.items():
    r = status_by_label[label]
    assert r['visual_review_status'] == 'PENDING', f'Label was previously marked reviewed: {label}'
    r['visual_review_status'] = 'VISUAL_REVIEW_REVISED' if info['final_label_sha256'] != info['base_label_sha256'] else 'VISUAL_REVIEW_NO_CHANGE'
    r['label_action'] = f"reviewed by worker {info['worker']} in {info['patch_zip']}; QC PASS"
    cnt = validate_yolo((DATASET_DIR / label).read_bytes(), label)
    r['box_count'] = str(sum(cnt.values()))
    r['classes_present'] = ','.join(CLASSES[i] for i in sorted(cnt))

with open(status_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(status_rows[0]))
    writer.writeheader()
    writer.writerows(status_rows)

training_counts = Counter()
validation_counts = Counter()
for entry in full_manifest:
    label = entry['label_path']
    cnt = validate_yolo((DATASET_DIR / label).read_bytes(), label)
    assert sha256_file(DATASET_DIR / label) == entry['label_sha256'], f'Manifest mismatch: {label}'
    if entry['split'] == 'train':
        training_counts.update(cnt)
    elif entry['split'] == 'val':
        validation_counts.update(cnt)
        assert reserved_hashes[label] == (entry['image_sha256'], entry['label_sha256'])
        assert sha256_file(DATASET_DIR / entry['image_path']) == entry['image_sha256']

with open(DATASET_DIR / 'manifest.json', 'w', encoding='utf-8') as f:
    json.dump(full_manifest, f, indent=2)

report_path = DATASET_DIR / 'merge_report.json'
with open(report_path, encoding='utf-8') as f:
    report = json.load(f)
report['class_boxes_by_split']['train'] = {CLASSES[i]: training_counts[i] for i in range(7)}
report['annotation_audit_progress'] = {'visually_reviewed_originals': 320 + len(changes),
                                        'pending_originals': 1252 - len(changes),
                                        'reviewed_worker_changes': len(changes),
                                        'independently_human_verified': False}
report['annotation_audit_status'] = 'PROVISIONAL — AI/worker-reviewed, independent human gold-standard verification not established'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

with open(DATASET_DIR / 'worker_patch_merge_audit.json', 'w', encoding='utf-8') as f:
    json.dump({'created_utc': datetime.now(timezone.utc).isoformat(),
               'base_sha256': BASE_SHA256, 'total_reviewed': 320 + len(changes),
               'remaining_unchecked': 1252 - len(changes),
               'patches': patch_log, 'replacements': review_info}, f, indent=2)

cfg = {'path': str(DATASET_DIR), 'train': 'images/train', 'val': 'images/val',
       'test': 'images/test', 'names': {i: c for i, c in enumerate(CLASSES)}}
with open(DATASET_DIR / 'data.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('TRAINING LABELS:', {CLASSES[i]: training_counts[i] for i in range(7)})
print('VALIDATION LABELS:', {CLASSES[i]: validation_counts[i] for i in range(7)})
assert all(validation_counts[i] == 0 for i in (3, 4, 6))
print(f'Originals inspected: {320 + len(changes)} / 1572, pending: {1252-len(changes)}')
print('✅ Reserved 97 validation and 71 test files are unmodified.')


## 5. Train YOLO11s

Default is a **hard stop** unless all assigned source images have reviewed worker labels with `qc_status=PASS`. To run a *clearly provisional* experiment before that, set `ALLOW_PROVISIONAL_TRAINING = True` in the setup cell and rerun from the top. Do **not** report provisional validation mAP as independent seven-class accuracy.

This trains **one unified 7-class detector** from pretrained YOLO11s weights, on a Colab GPU. 60 epochs, 800px images, AdamW, batch 4; adjust batch to 2 if GPU memory is insufficient.

In [ ]:
import torch
from ultralytics import YOLO
import ultralytics

print('PyTorch:', torch.__version__, '| Ultralytics:', ultralytics.__version__)
assert torch.cuda.is_available(), 'Enable a GPU: Runtime → Change runtime type → T4 GPU.'
assert len(full_manifest) == 2261

outstanding = 1252 - len(changes)
if REQUIRE_ALL_1252_REVIEWED and outstanding > 0 and not ALLOW_PROVISIONAL_TRAINING:
    raise RuntimeError(
        f'NOT TRAINING: {outstanding} of 1,252 worker-assigned original images still lack '
        'a reviewed, QC-PASS annotation patch. Add worker patch ZIPs to reviewed_patches/ '
        'or explicitly set ALLOW_PROVISIONAL_TRAINING=True for exploratory results only.'
    )
if outstanding > 0:
    print(f'⚠️ PROVISIONAL RUN: {outstanding} unchecked originals; annotations are not fully verified.')
print('⚠️ Validation set lacks labeled classes 3, 4 and 6. Its mAP is NOT a valid 7-class benchmark.')

model = YOLO(MODEL_WEIGHTS)
train_results = model.train(
    data=str(DATASET_DIR / 'data.yaml'),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    optimizer='AdamW',
    lr0=0.0005,
    patience=20,
    workers=2,
    seed=57,
    deterministic=True,
    amp=True,
    cos_lr=True,
    degrees=7.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    mosaic=0.8,
    mixup=0.05,
    close_mosaic=10,
    save_period=5,
    project=str(ROOT / 'training_runs'),
    name=RUN_NAME,
    exist_ok=False,
    plots=True,
)
run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / 'weights' / 'best.pt'
assert best_pt.is_file(), f'Expected trained checkpoint not found: {best_pt}'
print('✅ Training complete. Best checkpoint:', best_pt)


## 6. Save data provenance and export checkpoints

Saves the exact training class counts, worker patch audit, model checkpoint paths, and an optional dataset snapshot ZIP. Training runs persist in your Drive even if the Colab tab closes. **Never evaluate the reserved test split** until its seven-class annotations have been separately reviewed.

In [ ]:
# Run this AFTER training. The Drive training_runs/ folder already holds best.pt, last.pt and plots.
summary = {
    'model': MODEL_WEIGHTS, 'epochs': EPOCHS, 'imgsz': IMAGE_SIZE,
    'batch': BATCH_SIZE, 'run_directory': str(run_dir),
    'checkpoint': str(best_pt),
    'base_dataset_sha256': BASE_SHA256,
    'worker_reviewed': len(changes), 'originals_pending': 1252 - len(changes),
    'training_boxes': {CLASSES[i]: training_counts[i] for i in range(7)},
    'validation_boxes': {CLASSES[i]: validation_counts[i] for i in range(7)},
    'validation_complete_for_seven_classes': False,
    'independently_verified_seven_class_test': False,
    'created_utc': datetime.now(timezone.utc).isoformat(),
}
with open(run_dir / 'DATASET_AND_TRAINING_PROVENANCE.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
shutil.copy2(DATASET_DIR / 'worker_patch_merge_audit.json', run_dir / 'worker_patch_merge_audit.json')
shutil.copy2(DATASET_DIR / 'merge_report.json', run_dir / 'merge_report.json')
print('Saved training run:', run_dir)
print('Best weights:', best_pt)

# Optional: archive the exact merged dataset including worker patches for reproducibility.
SAVE_MERGED_DATASET_ZIP = False
if SAVE_MERGED_DATASET_ZIP:
    target = ROOT / 'YOLO_V5_merged_reviewed_labels.zip'
    with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=5) as z:
        for file in DATASET_DIR.rglob('*'):
            if file.is_file():
                z.write(file, arcname='v5_original_plus_v4/' + str(file.relative_to(DATASET_DIR)))
    assert zipfile.ZipFile(target).testzip() is None
    print('Saved merged dataset ZIP:', target)


### Scientific limitations

- The original 1,252 worker images are **not newly added photographs**: they are already in V5. The completed worker outputs replace their provisional **labels**.
- **320** old originals have provisional AI review; other original images still require completed worker QA.
- Existing V4 training photos may still have missing cross-class boxes (not fixed by the three original-source worker assignments).
- The 97-image V5 validation split has no labeled examples for `hinged_container`, `drink_carton`, or `bottle`; train selection and mAP from this split do **not** establish performance on those classes. Wrapper validation is sparse.
- Do not evaluate on the 71 reserved test images until a genuinely independent, seven-class complete test split is established.
- Hash checks confirm integrity and correct worker ownership, **not the semantic correctness** of each box.

Ultralytics YOLO11 Python API: https://docs.ultralytics.com/models/yolo11/ and https://docs.ultralytics.com/modes/train/ .